In [ ]:
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

from input import input
import config
from model import generics, single_ml_model_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# === CELULA DE CONFIGURACAO -- Tarefa 6 (mesmo padrao ja usado pelos
# notebooks de FS de MLP single desde a Tarefa 5) ===
# Edite aqui para uma nova rodada: series, experiment_id, grid. Ver
# RUNBOOK.md Secao 1 (series/lag_size) e Secao 3 (convencao de experiment_id).
#
# Tarefa 6 do PLANO_ARQUITETURA.md: generalizacao do arsenal de Feature
# Selection para SVR single, via SKlearnModel -- confirmado por pre-check
# (Tarefa 6) que a mesma integracao Pipeline([selector, estimador]) que ja
# funcionava para MLPRegressor (Tarefa 5) funciona identica para SVR, sem
# NENHUMA mudanca em single_ml_model_exp.py/grid_search_exp.py.
#
# Diferencas REAIS em relacao aos notebooks de MLP single (nao um
# copy-paste ingenuo): diff_kpss=True e model_exec=1 (SVR e deterministico
# -- CLAUDE.md Secao 3.4, mesma convencao do baseline svr_exec.ipynb) --
# NAO uniformizar com o model_exec=10 do MLP. Grid de hiperparametros
# (C/gamma/kernel/epsilon/tol) extraido diretamente de svr_exec.ipynb, para
# manter paridade com o baseline.
#
# strategy fixa nesta execucao ('rfecv') -- numero de features decidido pela CV interna do RFECV, SEM selector__k (Tarefa 4).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='rfecv')),
    ('estimator', SVR(max_iter=100000)),
])

# Series a rodar nesta execucao (FS_DEV_SERIES por padrao -- tests/model/conftest.py).
# lag_size='auto' medido na Tarefa 6 (pre-check, com diff_kpss=True): resolve
# para os MESMOS valores ja medidos para MLP single e para o hibrido
# ARIMA-MLP (airlines=20, austres=1, coloradoRiver=16, sunspot=9) -- PACF
# roda sobre a serie ANTES da diferenciacao KPSS, entao diff_kpss nao afeta
# esse valor (confirmado com dado real, nao assumido -- ver
# tests/model/test_single_ml_model_exp.py::TestSKlearnModelAcceptsPipelineWithSVR).
# df_train tem 1 linha a menos que o MLP single para a mesma serie (ex.
# airlines: 95 aqui vs. 96 no MLP) -- diff_kpss=True consome uma linha extra
# na diferenciacao, diferenca estrutural esperada.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md) -- nunca reaproveitar.
# Mesma convencao da Tarefa 5: chamados_v4_fs_svr_<metodo>.
experiment_id = 'chamados_v4_fs_svr_rfecv'
# Caminhos derivados de experiment_id, computados uma vez aqui e reusados nas
# celulas 3/4/5 (mesmo padrao ja usado pelos notebooks de MLP single).
experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py precisam disso. Baseline SVR single usa model_name='svr' (sem
# prefixo de horizon, adicionado automaticamente por
# GridSearch/generics.format_names) -- aqui 'svrrfecv' vira
# '1svrrfecv' no nome do .pkl, mesma logica de 'mlprfembedded' ->
# '1mlprfembedded' no MLP single.
model_name = 'svrrfecv'
normalize = True
force = False
# SVR e deterministico (CLAUDE.md Secao 3.4) -- model_exec=1, igual ao
# baseline svr_exec.ipynb. NAO usar 10 (isso e especifico de MLP, que e
# estocastico por inicializacao de pesos).
model_exec = 1

experiment_params = {
    'diff_kpss': True,
    'horizon': 1,
    'type_filter': None,
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
# Grid extraido de svr_exec.ipynb (C/gamma/kernel/epsilon/tol), para manter
# paridade de grid com o baseline.
model_parameters = {
    'estimator__C': [10, 100, 1000],
    'estimator__gamma': ['auto'],
    'estimator__kernel': ['rbf'],
    'estimator__epsilon': [0.1, 0.01, 0.001],
    'estimator__tol': [0.001],
   }

In [ ]:
# Sanity-check exigido pela Tarefa 2/5/6: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__C', 'estimator__kernel'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

# Checagem de identidade (mesmo padrao ja usado pelos notebooks de MLP
# single/ARIMA-MLP, achado de code-review da Tarefa 3.2): trava que a
# 'strategy' do seletor (celula 1) e o experiment_id/model_name declarados
# na MESMA celula continuam consistentes entre si.
strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"experiment_id={experiment_id!r} -- confirme que voce nao mudou 'strategy' "
    "na celula de configuracao sem atualizar experiment_id/model_name."
)
assert strategy_slug in model_name, (
    f"strategy={model.named_steps['selector'].strategy!r} nao aparece em "
    f"model_name={model_name!r} -- confirme consistencia."
)
print(f"OK -- strategy={model.named_steps['selector'].strategy!r} consistente com experiment_id={experiment_id!r} e model_name={model_name!r}")

In [ ]:
# Chamado direto via GridSearch(...).execution() por serie, em vez de
# grid_seach_multiple_bases(), para nao depender/mutar config.BASE_NAME_LIST
# (mesmo motivo ja documentado nos notebooks de MLP single/ARIMA-MLP). Nao ha
# celula de "copia do ARIMA" -- SKlearnModel nao depende de nenhum modelo
# linear pre-treinado, opera direto sobre a serie bruta.
# use_val_slipt_for_prev=True e explicito porque o default de GridSearch
# (False) diverge do default do wrapper grid_seach_multiple_bases (True) --
# sem isso, o refit final perderia o val_size e os .pkl ficariam sem
# val_metrics, quebrando a paridade com o baseline original (svr em
# chamados/).
for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        single_ml_model_exp.SKlearnModel,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()

In [ ]:
# Gera o CSV de metricas (agregado + detalhado) direto no notebook -- mesma
# funcao usada pela CLI (src/utils/export_metrics_to_csv.py), so o ponto de
# entrada muda (mesmo padrao notebook-only ja usado pelo MLP single/hibrido).
from utils.export_metrics_to_csv import run_export_metrics_to_csv

metrics_output = experiment_dir_results / 'metrics.csv'
df_metrics = run_export_metrics_to_csv(
    experiment_dir,
    metrics_output,
    detail=True,
)
df_metrics

In [ ]:
# Gera o CSV de features selecionadas (agregado + detalhado) direto no
# notebook -- mesma funcao usada pela CLI (src/utils/export_selected_features.py),
# so o ponto de entrada muda (mesmo padrao notebook-only ja usado pelo MLP single/hibrido).
from utils.export_selected_features import run_export_selected_features

features_output = experiment_dir_results / 'selected_features.csv'
df_features = run_export_selected_features(
    experiment_dir,
    features_output,
    detail=True,
)
df_features